In [ ]:
import tifffile as tf
import matplotlib.pyplot as plt
import numpy as np
import pickle
from pathlib import Path
import pandas as pd
import cv2
import napari
from tqdm import tqdm
from skimage.measure import regionprops
from matplotlib.colors import to_rgba
from scipy.spatial import ConvexHull

In [ ]:
!pip install distinctipy

In [ ]:
import distinctipy
def create_color_dict(labels_unique, excluded_colors=None):
    color_arr = distinctipy.get_colors(len(labels_unique), exclude_colors=excluded_colors)
    cell_type_color_dict = {}
    for i in range(len(labels_unique)):
        cell_type_color_dict[labels_unique[i]] = color_arr[i]
    return cell_type_color_dict

In [ ]:
%gui qt 

In [ ]:
from scipy.interpolate import splprep, splev
def smooth_contour(contour, smoothing=0.5, num_points=500):
    x, y = contour[:, 1], contour[:, 0]  
    
    if not np.allclose(contour[0], contour[-1]):
        x = np.r_[x, x[0]]
        y = np.r_[y, y[0]]

    tck, _ = splprep([x, y], s=smoothing, per=True)

    u_new = np.linspace(0, 1, num_points)
    x_new, y_new = splev(u_new, tck)
    
    return np.column_stack([y_new, x_new])


In [ ]:
def overlay_cells_in_napari(cell_segmentation_mask,
                            cell_cluster_location_df,
                            cell_type_color_dict,
                            image_background,
                            alpha):

    viewer = napari.Viewer()

    viewer.add_labels(cell_segmentation_mask, name="mask")

    image_background = (image_background * 255).astype(np.uint8)
    image_background = cv2.cvtColor(image_background, cv2.COLOR_RGB2RGBA)

    white_mask = np.all(image_background[:, :, :3] == 255, axis=-1)
    image_background[..., 3] = np.where(
        white_mask,
        255,
        int(alpha * 255)
    )

    viewer.add_image(image_background)

    regions = {r.label: r for r in regionprops(cell_segmentation_mask)}

    shapes_data = []
    face_colors = []
    edge_colors = []

    for _, row in tqdm(cell_cluster_location_df.iterrows()):
        cell_id = row['Cell ID']
        cell_type = row['Cell Type']

        if cell_id not in regions:
            continue

        region = regions[cell_id]
        coords = region.coords  

        
        hull = ConvexHull(coords)
        polygon = coords[hull.vertices]

        polygon = polygon.astype(np.float32)

        shapes_data.append(polygon)

        base_color = cell_type_color_dict[cell_type]
        face_colors.append(to_rgba(base_color, alpha=0.8))
        edge_colors.append(to_rgba(base_color, alpha=1.0))

    if len(shapes_data) > 0:
        viewer.add_shapes(
            shapes_data,
            shape_type='polygon',
            edge_width=2,
            face_color=face_colors,
            edge_color=edge_colors,
            name="cells"
        )

    viewer.theme = 'light'

    return viewer

In [ ]:
key = '/Multiplex 12/Rqo20'

In [ ]:
registration_path = 'Y:/coskun-lab/Aditi/01/Results/registration/' + key #+ '/1x'

In [ ]:
cell_clustering_path = 'Y:/coskun-lab/Aditi/01/Results/cell_segmentation_clustering/' + key + '/1x'

In [ ]:
fibrosis_path = 'Y:/coskun-lab/Aditi/01/Results/fibrosis_segmentation/' + key 

In [ ]:
with open(cell_clustering_path + '/segmentation_mask_merged_within_contour_for_pixel_clustering.pickle', 'rb') as f:
    cell_segmentation_mask = pickle.load(f)

In [ ]:
with open(registration_path + '/roi_coord_dict.pickle', 'rb') as f: 
    roi = pickle.load(f)

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_path + '/cell_cluster_location_merged_unclassified_within_contour.csv').drop(columns=['Unnamed: 0'])

In [ ]:
with open(fibrosis_path + '/wsi_labeled.pickle', 'rb') as f:
    wsi_labeled = pickle.load(f)

In [ ]:
with open(fibrosis_path + '/cluster_color_dict.pickle', 'rb') as f:
    cluster_color_dict = pickle.load(f)

In [ ]:
with open(cell_clustering_path + '/cell_type_color_dict.pickle', 'rb') as f: 
    cell_type_color_dict = pickle.load(f)

In [ ]:
wsi_labeled[np.isnan(wsi_labeled)] = 0

In [ ]:
wsi_labeled = wsi_labeled.astype(int)

In [ ]:
cluster_color_dict[0] = (1.0, 1.0, 1.0)

In [ ]:
max_pixel_value = int(np.max(wsi_labeled))
color_array = np.zeros((max_pixel_value + 1, 3))

for key, value in cluster_color_dict.items():
    color_array[key] = value

wsi_labeled_converted = color_array[wsi_labeled]

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

cell_types = list(set(list(cell_type_color_dict.keys())))
cell_types.remove('Unclassified')

fig, ax = plt.subplots(1)
i = 0

patches = [
    Line2D([], [], 
           color=color, 
           marker='o', 
           linestyle='None',
           markersize=12,
           label=label)
    for label, color in cell_type_color_dict.items() if label in cell_types
]

ax.legend(
    patches,
    list(cell_types),
    loc='center',
    frameon=False,
    ncol=len(cell_type_color_dict)/len(cell_type_color_dict), labelspacing=1
)

ax.axis('off')


In [ ]:
cell_types = list(set(list(cell_type_color_dict.keys())))
cell_types.remove('Unclassified')
colors_to_exclude = list(cluster_color_dict.values())
colors_to_exclude.extend([(0, 0, 0), (1, 1, 1)])
cell_type_color_dict_fibrosis_excluded = create_color_dict(cell_types, colors_to_exclude)

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
fig, ax = plt.subplots(1)
i = 0

patches = [
    Line2D([], [], 
           color=color, 
           marker='o', 
           linestyle='None',
           markersize=12,
           label=label)
    for label, color in cell_type_color_dict_fibrosis_excluded.items()
]

ax.legend(
    patches,
    list(cell_type_color_dict_fibrosis_excluded.keys()),
    loc='center',
    frameon=False,
    ncol=len(cell_type_color_dict_fibrosis_excluded)/len(cell_type_color_dict_fibrosis_excluded), labelspacing=1
)

ax.axis('off')


In [ ]:
with open(fibrosis_path + '/cell_type_color_dict_fibrosis_excluded.pickle', 'wb') as f: 
    pickle.dump(cell_type_color_dict_fibrosis_excluded, f)

In [ ]:
wsi_labeled_converted = np.pad(wsi_labeled_converted, ((0, 0), (0, cell_segmentation_mask.shape[1]-wsi_labeled_converted.shape[1]), (0, 0)))

In [ ]:
for name, coord in roi.items():
    cell_cluster_location_for_region = cell_cluster_location_df[(cell_cluster_location_df['Y'].ge(coord[0])) & (cell_cluster_location_df['Y'].le(coord[1])) & (cell_cluster_location_df['X'].ge(coord[2])) & (cell_cluster_location_df['X'].le(coord[3]))]
    cell_cluster_location_for_region = cell_cluster_location_for_region[cell_cluster_location_for_region['Cell Type'] != 'Unclassified']
    cell_segmentation_mask_region = cell_segmentation_mask.copy()
    cell_segmentation_mask_region  = cell_segmentation_mask_region[coord[0]:coord[1], coord[2]:coord[3]]
    viewer = overlay_cells_in_napari(cell_segmentation_mask_region, cell_cluster_location_for_region, cell_type_color_dict_fibrosis_excluded, wsi_labeled_converted[coord[0]:coord[1], coord[2]:coord[3]], 0.4)